# Ajnyana — Architecture

nanoGPT-style decoder-only transformer (Karpathy).

- Standard MHA + causal mask
- Pre-norm (LayerNorm sebelum attn & MLP)
- GELU activation
- Learned positional embeddings
- Weight tying (embedding ↔ lm_head)
- Target: ~10M params

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import os
from dataclasses import dataclass

os.makedirs('../scripts', exist_ok=True)

print(f'PyTorch: {torch.__version__}')
print(f'Device:  {"cuda" if torch.cuda.is_available() else "cpu"}')

PyTorch: 2.11.0
Device:  cpu


## 1. Config

In [2]:
@dataclass
class GPTConfig:
    vocab_size : int   = 16_000
    block_size : int   = 512      # max sequence length
    n_layer    : int   = 6
    n_head     : int   = 4
    n_embd     : int   = 256      # dim=256 → ~9M params (dim=512 → ~27M)
    dropout    : float = 0.1
    bias       : bool  = False    # no bias, like Karpathy

cfg = GPTConfig()
print(cfg)

GPTConfig(vocab_size=16000, block_size=512, n_layer=6, n_head=4, n_embd=256, dropout=0.1, bias=False)


## 2. Model

In [3]:
class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg.n_embd % cfg.n_head == 0
        self.n_head  = cfg.n_head
        self.n_embd  = cfg.n_embd
        self.dropout = cfg.dropout
        self.c_attn  = nn.Linear(cfg.n_embd, 3 * cfg.n_embd, bias=cfg.bias)
        self.c_proj  = nn.Linear(cfg.n_embd, cfg.n_embd,     bias=cfg.bias)
        self.attn_drop = nn.Dropout(cfg.dropout)
        self.resid_drop = nn.Dropout(cfg.dropout)
        # causal mask
        self.register_buffer('bias', torch.tril(torch.ones(cfg.block_size, cfg.block_size))
                             .view(1, 1, cfg.block_size, cfg.block_size))

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        head_dim = C // self.n_head
        q = q.view(B, T, self.n_head, head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(head_dim))
        att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.attn_drop(att)
        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.c_proj(y))


class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.c_fc   = nn.Linear(cfg.n_embd, 4 * cfg.n_embd, bias=cfg.bias)
        self.c_proj = nn.Linear(4 * cfg.n_embd, cfg.n_embd, bias=cfg.bias)
        self.drop   = nn.Dropout(cfg.dropout)

    def forward(self, x):
        return self.drop(self.c_proj(F.gelu(self.c_fc(x))))


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.n_embd, bias=cfg.bias)
        self.attn = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.n_embd, bias=cfg.bias)
        self.mlp  = MLP(cfg)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.transformer = nn.ModuleDict(dict(
            wte  = nn.Embedding(cfg.vocab_size, cfg.n_embd),
            wpe  = nn.Embedding(cfg.block_size, cfg.n_embd),
            drop = nn.Dropout(cfg.dropout),
            h    = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)]),
            ln_f = nn.LayerNorm(cfg.n_embd, bias=cfg.bias),
        ))
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        # weight tying
        self.transformer.wte.weight = self.lm_head.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.cfg.block_size
        pos = torch.arange(T, device=idx.device)
        x = self.transformer.drop(
            self.transformer.wte(idx) + self.transformer.wpe(pos)
        )
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.cfg.block_size else idx[:, -self.cfg.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')
            probs = F.softmax(logits, dim=-1)
            next_tok = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_tok), dim=1)
        return idx

print('Model defined OK')

Model defined OK


## 3. Parameter Count

In [4]:
model = GPT(cfg)

total  = sum(p.numel() for p in model.parameters())
unique = sum(p.numel() for p in set(model.parameters()))  # dedup weight-tied

print(f'Total params (with weight tie counted once): {unique:,}')
print(f'Total params (raw):                          {total:,}')
print()

# Per-component breakdown
wte  = sum(p.numel() for p in model.transformer.wte.parameters())
wpe  = sum(p.numel() for p in model.transformer.wpe.parameters())
blks = sum(p.numel() for p in model.transformer.h.parameters())
lnf  = sum(p.numel() for p in model.transformer.ln_f.parameters())
print(f'  Embedding (wte, tied w/ lm_head): {wte:>10,}')
print(f'  Positional embedding (wpe):       {wpe:>10,}')
print(f'  {cfg.n_layer}x Transformer blocks:        {blks:>10,}')
print(f'  Final LayerNorm:                  {lnf:>10,}')

Total params (with weight tie counted once): 8,948,992
Total params (raw):                          8,948,992

  Embedding (wte, tied w/ lm_head):  4,096,000
  Positional embedding (wpe):          131,072
  6x Transformer blocks:         4,721,664
  Final LayerNorm:                         256


## 4. Forward Pass Test

In [5]:
model.eval()
B, T = 2, 128

x = torch.randint(0, cfg.vocab_size, (B, T))
y = torch.randint(0, cfg.vocab_size, (B, T))

logits, loss = model(x, targets=y)

print(f'Input:   {x.shape}')
print(f'Logits:  {logits.shape}  (expect [{B}, {T}, {cfg.vocab_size}])')
print(f'Loss:    {loss.item():.4f}  (expect ~{math.log(cfg.vocab_size):.2f} at init)')
print()

# Generation test
prompt = torch.zeros((1, 1), dtype=torch.long)
out = model.generate(prompt, max_new_tokens=20, temperature=1.0, top_k=50)
print(f'Generated token IDs: {out[0].tolist()}')
print('Forward pass OK ✅')

Input:   torch.Size([2, 128])
Logits:  torch.Size([2, 128, 16000])  (expect [2, 128, 16000])
Loss:    9.7411  (expect ~9.68 at init)

Generated token IDs: [0, 15377, 5968, 5690, 626, 2292, 14533, 14339, 10009, 12837, 2086, 878, 11724, 7001, 2358, 6527, 3837, 15282, 11665, 12675, 10057]
Forward pass OK ✅


## 5. Save model.py

In [6]:
model_py = '''
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass


@dataclass
class GPTConfig:
    vocab_size : int   = 16_000
    block_size : int   = 512
    n_layer    : int   = 6
    n_head     : int   = 4
    n_embd     : int   = 256
    dropout    : float = 0.1
    bias       : bool  = False


class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg.n_embd % cfg.n_head == 0
        self.n_head  = cfg.n_head
        self.n_embd  = cfg.n_embd
        self.dropout = cfg.dropout
        self.c_attn  = nn.Linear(cfg.n_embd, 3 * cfg.n_embd, bias=cfg.bias)
        self.c_proj  = nn.Linear(cfg.n_embd, cfg.n_embd,     bias=cfg.bias)
        self.attn_drop  = nn.Dropout(cfg.dropout)
        self.resid_drop = nn.Dropout(cfg.dropout)
        self.register_buffer("bias", torch.tril(torch.ones(cfg.block_size, cfg.block_size))
                             .view(1, 1, cfg.block_size, cfg.block_size))

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        head_dim = C // self.n_head
        q = q.view(B, T, self.n_head, head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, head_dim).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(head_dim))
        att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = self.attn_drop(att)
        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.c_proj(y))


class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.c_fc   = nn.Linear(cfg.n_embd, 4 * cfg.n_embd, bias=cfg.bias)
        self.c_proj = nn.Linear(4 * cfg.n_embd, cfg.n_embd, bias=cfg.bias)
        self.drop   = nn.Dropout(cfg.dropout)

    def forward(self, x):
        return self.drop(self.c_proj(F.gelu(self.c_fc(x))))


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1  = nn.LayerNorm(cfg.n_embd, bias=cfg.bias)
        self.attn = CausalSelfAttention(cfg)
        self.ln2  = nn.LayerNorm(cfg.n_embd, bias=cfg.bias)
        self.mlp  = MLP(cfg)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.transformer = nn.ModuleDict(dict(
            wte  = nn.Embedding(cfg.vocab_size, cfg.n_embd),
            wpe  = nn.Embedding(cfg.block_size, cfg.n_embd),
            drop = nn.Dropout(cfg.dropout),
            h    = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)]),
            ln_f = nn.LayerNorm(cfg.n_embd, bias=cfg.bias),
        ))
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.cfg.block_size
        pos = torch.arange(T, device=idx.device)
        x = self.transformer.drop(
            self.transformer.wte(idx) + self.transformer.wpe(pos)
        )
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.cfg.block_size else idx[:, -self.cfg.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float("-inf")
            probs = F.softmax(logits, dim=-1)
            next_tok = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_tok), dim=1)
        return idx
'''.strip()

with open('../scripts/model.py', 'w') as f:
    f.write(model_py)

print('Saved: scripts/model.py')

Saved: scripts/model.py


## Next

Architecture ✅ → `04_train.ipynb` — tokenize corpus ke binary, training loop, checkpoint save.